# LeetCode 761 — Special Binary String bằng Python

- Link: https://leetcode.com/problems/special-binary-string/
- Độ khó: **Hard**
- Chủ đề: Đệ quy, Chia để trị, Chuỗi ngoặc đúng (Dyck word), Sắp xếp chuỗi, Cây

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def makeLargestSpecial(self, s: str) -> str:
```

## 1. Giới thiệu & mục tiêu học

Một "chuỗi nhị phân đặc biệt" thực chất là **chuỗi ngoặc đúng** trá hình: `1` là `(`, `0` là `)`. Phép "hoán đổi hai chuỗi đặc biệt liền kề" chính là **đổi chỗ hai khối ngoặc anh em**. Khi nhìn ra điều này, bài Hard trở thành một đệ quy rất ngắn: **tách thành các khối, giải đệ quy bên trong, sắp các khối giảm dần, ghép lại**.

Sau notebook này bạn sẽ:

1. Nhận ra cấu trúc **chuỗi ngoặc / cây** ẩn sau định nghĩa "đặc biệt".
2. Viết lời giải BFS vét cạn mọi trạng thái (baseline) trên chuỗi ngắn.
3. Hiểu cách **phân rã** một chuỗi đặc biệt thành các **khối nguyên thuỷ** `1 + A + 0`.
4. Cài đặt lời giải **đệ quy chia để trị** và bản **dùng stack (không đệ quy)**.
5. Chứng minh vì sao **sắp các khối theo thứ tự giảm dần** cho chuỗi lớn nhất.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Một chuỗi nhị phân là **đặc biệt** nếu:

1. Số ký tự `0` bằng số ký tự `1`.
2. **Mọi tiền tố** có số `1` **lớn hơn hoặc bằng** số `0`.

Cho chuỗi đặc biệt `s`. Một **bước đi**: chọn hai chuỗi con **đặc biệt, khác rỗng, liền kề nhau** trong `s` rồi **đổi chỗ** chúng. Hãy trả về chuỗi **lớn nhất theo thứ tự từ điển** có thể đạt được sau một số bước bất kỳ.

**Ví dụ 1**

```text
Input: s = "11011000"
Output: "11100100"
Giải thích: hai chuỗi con "10" (tại s[1]) và "1100" (tại s[3]) liền kề, đổi chỗ được "11100100".
Đây là chuỗi lớn nhất có thể.
```

**Ví dụ 2**

```text
Input: s = "10"
Output: "10"
```

**Ràng buộc**

- `1 <= s.length <= 50`
- `s[i]` là `'0'` hoặc `'1'`.
- `s` là chuỗi nhị phân đặc biệt.

In [1]:
def is_special(t):
    bal = 0
    for c in t:
        bal += 1 if c == '1' else -1
        if bal < 0:
            return False
    return bal == 0 and len(t) > 0

EXAMPLES = [("11011000", "11100100"), ("10", "10")]

for t in ["10", "1100", "1010", "0110", "11011000", "110"]:
    print(t, is_special(t))

10 True
1100 True
1010 True
0110 False
11011000 True
110 False


## 3. Kiến thức nền

### 3.1. Số dư (balance) và chuỗi ngoặc đúng

Đi từ trái sang phải, `bal += 1` với `1` và `bal -= 1` với `0`. Chuỗi đặc biệt ⇔ `bal` không bao giờ âm và kết thúc bằng 0 — giống hệt điều kiện chuỗi ngoặc `(` `)` hợp lệ.

In [2]:
def to_paren(t):
    return t.replace('1', '(').replace('0', ')')

s = "11011000"
bal, hist = 0, []
for c in s:
    bal += 1 if c == '1' else -1
    hist.append(bal)
print(to_paren(s), hist)

(()(())) [1, 2, 1, 2, 3, 2, 1, 0]


### 3.2. So sánh chuỗi theo thứ tự từ điển trong Python

`a < b` so sánh từng ký tự từ trái sang; ký tự đầu tiên khác nhau quyết định. Nếu một chuỗi là tiền tố của chuỗi kia thì chuỗi ngắn hơn nhỏ hơn. `'1' > '0'` nên muốn chuỗi lớn thì muốn `1` xuất hiện **càng sớm càng tốt**.

In [3]:
print("1100" > "1010", "10" < "1010", sorted(["10", "1100", "1010"], reverse=True))

True True ['1100', '1010', '10']


### 3.3. Đệ quy trên cấu trúc lồng nhau

Hàm đệ quy xử lý một khối, gọi chính nó cho phần bên trong. Ví dụ: tính độ sâu lồng tối đa của chuỗi ngoặc.

In [4]:
def max_depth(t):
    best = bal = 0
    for c in t:
        bal += 1 if c == '1' else -1
        best = max(best, bal)
    return best

print(max_depth("11011000"), max_depth("10101010"), max_depth("11110000"))

3 1 4


### 3.4. BFS với `deque` và tập `visited`

Để duyệt mọi trạng thái đến được từ trạng thái ban đầu: hàng đợi `deque`, tập `seen` tránh lặp. Phần tử nào cũng được xử lý đúng một lần.

In [5]:
from collections import deque

def bfs_demo(start, neighbors):
    seen, q = {start}, deque([start])
    while q:
        u = q.popleft()
        for v in neighbors(u):
            if v not in seen:
                seen.add(v)
                q.append(v)
    return seen

# ví dụ: các số đến được từ 1 bằng phép x*2 hoặc x+3, giới hạn <= 20
print(sorted(bfs_demo(1, lambda x: [y for y in (2 * x, x + 3) if y <= 20])))

[1, 2, 4, 5, 7, 8, 10, 11, 13, 14, 16, 17, 19, 20]


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Phân rã thành khối nguyên thuỷ

Đi dọc chuỗi, mỗi lần `bal` về 0 ta cắt được một **khối nguyên thuỷ** (primitive). Ví dụ `s = "11011000"`: `bal` = 1,2,1,2,3,2,1,0 → chỉ về 0 ở cuối → cả chuỗi là **một** khối.

Một khối nguyên thuỷ luôn có dạng **`1` + (chuỗi đặc biệt hoặc rỗng) + `0`**: bỏ ký tự đầu và cuối, phần giữa vẫn đặc biệt.

- `"11011000"` = `1` + `"101100"` + `0`.
- `"101100"`: `bal` = 1,0 | 1,2,1,0 → hai khối `"10"` và `"1100"`.

### 4.2. Đổi chỗ = sắp xếp các khối anh em

Hai chuỗi con đặc biệt liền kề cùng "tầng" chính là hai **dãy khối anh em** liên tiếp. Đổi chỗ được thì ta có thể **hoán vị tuỳ ý** các khối anh em (mọi hoán vị là tích của các phép đổi chỗ kề nhau). Muốn chuỗi lớn nhất → đặt khối **lớn hơn** lên trước.

Với phần giữa `"101100"`: các khối `"10"`, `"1100"`. Sắp giảm dần: `"1100" > "10"` → `"110010"`. Bọc lại: `"1" + "110010" + "0"` = `"11100100"`. ✔

### 4.3. Biểu diễn cây

```text
s = 1 1 0 1 1 0 0 0
     ( ( ) ( ( ) ) )

gốc
└── khối "11011000"
     ├── "10"
     └── "1100"
          └── "10"
```

Mỗi nút = một khối nguyên thuỷ, con của nó = các khối bên trong. Ta được phép sắp lại **thứ tự các con** của mỗi nút. Đáp án = sắp con của mọi nút theo chuỗi giảm dần, từ dưới lên.

In [6]:
def split_primitives(t):
    res, bal, start = [], 0, 0
    for i, c in enumerate(t):
        bal += 1 if c == '1' else -1
        if bal == 0:
            res.append(t[start:i + 1])
            start = i + 1
    return res

print(split_primitives("11011000"))
print(split_primitives("101100"))
print(split_primitives("10110010"))

['11011000']
['10', '1100']
['10', '1100', '10']


## 5. Cách 1 — Brute Force: BFS qua mọi chuỗi đạt được

**Ý tưởng.** Từ chuỗi hiện tại, liệt kê mọi bộ `i < j < k` sao cho `t[i:j]` và `t[j:k]` đều đặc biệt; chuỗi mới là `t[:i] + t[j:k] + t[i:j] + t[k:]`. BFS qua mọi chuỗi đạt được và lấy `max`.

In [7]:
def moves(t):
    n = len(t)
    out = set()
    for i in range(n):
        for j in range(i + 2, n + 1, 2):
            if not is_special(t[i:j]):
                continue
            for k in range(j + 2, n + 1, 2):
                if is_special(t[j:k]):
                    out.add(t[:i] + t[j:k] + t[i:j] + t[k:])
    return out

def brute(s):
    seen, q = {s}, deque([s])
    while q:
        u = q.popleft()
        for v in moves(u):
            if v not in seen:
                seen.add(v)
                q.append(v)
    return max(seen)

for s, exp in EXAMPLES:
    print(s, "->", brute(s), "(kỳ vọng", exp, ")")
print("số trạng thái đạt được từ 11011000:", len(bfs_demo("11011000", moves)))

11011000 -> 11100100 (kỳ vọng 11100100 )
10 -> 10 (kỳ vọng 10 )
số trạng thái đạt được từ 11011000: 2


**Độ phức tạp.** Mỗi trạng thái sinh `O(n^3)` bộ `(i, j, k)`, mỗi kiểm tra `O(n)` → `O(n^4)` mỗi trạng thái. Số trạng thái có thể bùng nổ (số cách sắp các cây con — giai thừa). Với `n = 50` không khả thi.

**Nút thắt.** Ta khám phá mù quáng mọi hoán vị. Nhưng thật ra ta **biết trước** đích đến tốt nhất: khối lớn đứng trước. Cần hiểu cấu trúc để "nhảy thẳng" tới đáp án.

## 6. Bước đột phá

**Insight 1 — Chuỗi đặc biệt = dãy khối nguyên thuỷ.** Cắt tại các vị trí `bal == 0`: `s = P1 P2 ... Pm`, mỗi `Pi = 1 + Ai + 0` với `Ai` đặc biệt (hoặc rỗng).

**Insight 2 — Mọi chuỗi con đặc biệt đều "tôn trọng" cấu trúc.** Một chuỗi con đặc biệt hoặc là nằm trọn trong một `Ai`, hoặc là một **dãy liên tiếp các khối** `Pi ... Pj` (ở cùng tầng). Vì vậy một bước đi chỉ có thể: đổi chỗ hai dãy khối anh em liên tiếp, ở một tầng nào đó. Ký tự `1` mở và `0` đóng của mỗi khối **không bao giờ** bị tách rời.

**Insight 3 — Tối ưu độc lập từng tầng.** Làm phần trong `Ai` lớn nhất trước (đệ quy), rồi sắp các khối đã tối ưu theo thứ tự **giảm dần**.

**Insight 4 — Vì sao sắp chuỗi giảm dần là đúng.** Hai khối nguyên thuỷ khác nhau không bao giờ là tiền tố của nhau (một khối chỉ về `bal = 0` ở cuối), nên `X > Y` (từ điển) ⇒ `X + Y > Y + X`. Vậy đặt khối lớn hơn lên trước luôn tốt hơn.

**Quy tắc nếu… thì…**

- Nếu gặp `bal == 0` → vừa kết thúc một khối, xử lý đệ quy phần trong của nó.
- Nếu đã có danh sách khối → `sort(reverse=True)` rồi `''.join`.
- Nếu chuỗi rỗng → trả về rỗng (trường hợp cơ sở).

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Đệ quy chia để trị

**Giải thích.**

```text
solve(t):
    blocks = []
    duyệt t, mỗi khi bal về 0 tại i (khối t[start..i]):
        blocks.append('1' + solve(t[start+1 : i]) + '0')
    sắp blocks giảm dần, ghép lại
```

In [8]:
def solve_rec(t):
    blocks = []
    bal = start = 0
    for i, c in enumerate(t):
        bal += 1 if c == '1' else -1
        if bal == 0:
            blocks.append('1' + solve_rec(t[start + 1:i]) + '0')
            start = i + 1
    blocks.sort(reverse=True)
    return ''.join(blocks)

for s, exp in EXAMPLES:
    print(s, "->", solve_rec(s), exp)

11011000 -> 11100100 11100100
10 -> 10 10


**Code nộp LeetCode (Cách 2).**

In [9]:
class SolutionRec:
    def makeLargestSpecial(self, s: str) -> str:
        blocks = []
        bal = start = 0
        for i, c in enumerate(s):
            bal += 1 if c == '1' else -1
            if bal == 0:
                blocks.append('1' + self.makeLargestSpecial(s[start + 1:i]) + '0')
                start = i + 1
        blocks.sort(reverse=True)
        return ''.join(blocks)

print([SolutionRec().makeLargestSpecial(s) for s, _ in EXAMPLES])

['11100100', '10']


**Độ phức tạp.** Mỗi tầng đệ quy quét phần chuỗi của nó: tổng quét `O(n · d)` với `d` là độ sâu lồng (≤ `n/2`) → `O(n^2)`. Cắt chuỗi và sắp xếp cũng bị chặn bởi `O(n^2)` (thêm `log`). Với `n = 50` cực nhanh. Bộ nhớ `O(n^2)` trong trường hợp xấu cho các chuỗi con, độ sâu đệ quy `O(n)`.

### 7.2. Cách 3 — Dùng stack (không đệ quy), xây từ trong ra ngoài

**Giải thích.** Giống phân tích ngoặc bằng stack:

- Stack chứa các **danh sách khối** của mỗi tầng đang mở. Ban đầu `stack = [[]]` (tầng ngoài cùng).
- Gặp `1` (mở): đẩy một danh sách rỗng mới cho tầng bên trong.
- Gặp `0` (đóng): pop danh sách khối bên trong, sắp giảm dần, ghép → `inner`; khối hoàn chỉnh `'1' + inner + '0'` được thêm vào danh sách của tầng ngoài (đỉnh stack mới).
- Cuối cùng sắp danh sách tầng ngoài cùng và ghép.

In [10]:
def solve_stack(s):
    stack = [[]]
    for c in s:
        if c == '1':
            stack.append([])
        else:
            inner = stack.pop()
            inner.sort(reverse=True)
            stack[-1].append('1' + ''.join(inner) + '0')
    top = stack[0]
    top.sort(reverse=True)
    return ''.join(top)

for s, exp in EXAMPLES:
    print(s, "->", solve_stack(s), exp)

11011000 -> 11100100 11100100
10 -> 10 10


**Code nộp LeetCode (Cách 3).**

In [11]:
class SolutionStack:
    def makeLargestSpecial(self, s: str) -> str:
        stack = [[]]
        for c in s:
            if c == '1':
                stack.append([])
            else:
                inner = stack.pop()
                inner.sort(reverse=True)
                stack[-1].append('1' + ''.join(inner) + '0')
        return ''.join(sorted(stack[0], reverse=True))

print([SolutionStack().makeLargestSpecial(s) for s, _ in EXAMPLES])

['11100100', '10']


**Độ phức tạp.** Mỗi ký tự `0` tạo một khối có độ dài bằng cây con của nó; tổng độ dài các chuỗi được tạo = tổng kích thước cây con = `O(n · d)` → `O(n^2)` xấu nhất. Bộ nhớ `O(n^2)` xấu nhất, không có đệ quy.

### 7.3. Cách 4 — Dựng cây tường minh rồi chuẩn hoá (canonical form)

**Giải thích.** Đây là góc nhìn "đồ thị/cây": chuyển chuỗi thành **cây có thứ tự** (mỗi khối là một nút), rồi tính **dạng chuẩn** của cây: `canon(node) = '1' + join(sorted([canon(c) for c in children], reverse=True)) + '0'`. Kỹ thuật này giống bài **kiểm tra hai cây đồng cấu** (AHU encoding) — hai chuỗi đặc biệt biến đổi được thành nhau ⇔ hai cây có cùng dạng chuẩn.

In [12]:
class Node:
    def __init__(self):
        self.children = []

def build_tree(s):
    root = Node()
    path = [root]
    for c in s:
        if c == '1':
            nd = Node()
            path[-1].children.append(nd)
            path.append(nd)
        else:
            path.pop()
    return root

def canon(node):
    return ''.join(sorted(('1' + canon(ch) + '0' for ch in node.children), reverse=True))

def solve_tree(s):
    return canon(build_tree(s))

for s, exp in EXAMPLES:
    print(s, "->", solve_tree(s), exp)

11011000 -> 11100100 11100100
10 -> 10 10


**Code nộp LeetCode (Cách 4).**

In [13]:
class SolutionTree:
    def makeLargestSpecial(self, s: str) -> str:
        children = [[]]              # children[v] = danh sách con của nút v; nút 0 là gốc ảo
        path = [0]
        for c in s:
            if c == '1':
                children.append([])
                v = len(children) - 1
                children[path[-1]].append(v)
                path.append(v)
            else:
                path.pop()

        def canon(v):
            parts = ['1' + canon(u) + '0' for u in children[v]]
            parts.sort(reverse=True)
            return ''.join(parts)

        return canon(0)

print([SolutionTree().makeLargestSpecial(s) for s, _ in EXAMPLES])

['11100100', '10']


**Độ phức tạp.** Dựng cây `O(n)`, chuẩn hoá `O(n^2)` xấu nhất (do ghép chuỗi). Bộ nhớ `O(n)` cho cây + `O(n^2)` cho chuỗi trung gian.

**Ứng dụng phụ:** kiểm tra hai chuỗi đặc biệt `a`, `b` có biến đổi qua lại được không: `solve(a) == solve(b)`.

In [14]:
print(solve_tree("1100101100") == solve_tree("1011001100"))   # cùng tập khối -> True
print(solve_tree("110100") == solve_tree("111000"))           # cấu trúc khác -> False

True
False


## 8. TRACE từng bước

Trace **Cách 2** trên `s = "11011000"`.

| lời gọi | chuỗi | các khối tìm được (sau đệ quy) | sau khi sắp | trả về |
|---|---|---|---|---|
| solve("11011000") | `11011000` | 1 khối: `1 + solve("101100") + 0` | — | chờ |
| solve("101100") | `101100` | `1+solve("")+0` = `10`; `1+solve("10")+0` | — | chờ |
| solve("10") | `10` | `1+solve("")+0` = `10` | `[10]` | `10` |
| quay lại solve("101100") | | `[10, 1100]` | `[1100, 10]` | `110010` |
| quay lại gốc | | `[1 110010 0]` | | `11100100` |

In [15]:
def trace_rec(t, depth=0):
    pad = "    " * depth
    print(f"{pad}solve({t!r})")
    blocks = []
    bal = start = 0
    for i, c in enumerate(t):
        bal += 1 if c == '1' else -1
        if bal == 0:
            inner = t[start + 1:i]
            print(f"{pad}  khối {t[start:i + 1]!r}, phần trong {inner!r}")
            blocks.append('1' + trace_rec(inner, depth + 1) + '0')
            start = i + 1
    before = blocks[:]
    blocks.sort(reverse=True)
    res = ''.join(blocks)
    print(f"{pad}  các khối {before} -> sắp {blocks} -> trả {res!r}")
    return res

trace_rec("11011000")

solve('11011000')
  khối '11011000', phần trong '101100'
    solve('101100')
      khối '10', phần trong ''
        solve('')
          các khối [] -> sắp [] -> trả ''
      khối '1100', phần trong '10'
        solve('10')
          khối '10', phần trong ''
            solve('')
              các khối [] -> sắp [] -> trả ''
          các khối ['10'] -> sắp ['10'] -> trả '10'
      các khối ['10', '1100'] -> sắp ['1100', '10'] -> trả '110010'
  các khối ['11100100'] -> sắp ['11100100'] -> trả '11100100'


'11100100'

Trace **Cách 3 (stack)** trên `s = "1010110100"`: in stack sau mỗi ký tự.

In [16]:
def trace_stack(s):
    stack = [[]]
    for i, c in enumerate(s):
        if c == '1':
            stack.append([])
            act = "mở tầng mới"
        else:
            inner = stack.pop()
            inner.sort(reverse=True)
            blk = '1' + ''.join(inner) + '0'
            stack[-1].append(blk)
            act = f"đóng -> khối {blk!r}"
        print(f"i={i} c={c} {act:<26} stack={stack}")
    res = ''.join(sorted(stack[0], reverse=True))
    print("kết quả:", res)
    return res

trace_stack("1010110100")

i=0 c=1 mở tầng mới                stack=[[], []]
i=1 c=0 đóng -> khối '10'          stack=[['10']]
i=2 c=1 mở tầng mới                stack=[['10'], []]
i=3 c=0 đóng -> khối '10'          stack=[['10', '10']]
i=4 c=1 mở tầng mới                stack=[['10', '10'], []]
i=5 c=1 mở tầng mới                stack=[['10', '10'], [], []]
i=6 c=0 đóng -> khối '10'          stack=[['10', '10'], ['10']]
i=7 c=1 mở tầng mới                stack=[['10', '10'], ['10'], []]
i=8 c=0 đóng -> khối '10'          stack=[['10', '10'], ['10', '10']]
i=9 c=0 đóng -> khối '110100'      stack=[['10', '10', '110100']]
kết quả: 1101001010


'1101001010'

## 9. Code nộp LeetCode

Bản chọn: **Cách 2 (đệ quy)** — ngắn gọn, phản ánh đúng cấu trúc bài toán.

In [17]:
class Solution:
    def makeLargestSpecial(self, s: str) -> str:
        blocks = []                 # các khối nguyên thuỷ ở tầng này
        bal = start = 0
        for i, c in enumerate(s):
            bal += 1 if c == '1' else -1
            if bal == 0:            # s[start..i] là một khối "1 A 0"
                inner = self.makeLargestSpecial(s[start + 1:i])
                blocks.append('1' + inner + '0')
                start = i + 1
        blocks.sort(reverse=True)   # khối lớn hơn đứng trước
        return ''.join(blocks)

for s, exp in EXAMPLES:
    assert Solution().makeLargestSpecial(s) == exp
print("OK")

OK


Giải thích từng dòng:

- `bal`: số dư `1` trừ `0` từ `start`. Vì `s` (và mọi phần trong) là đặc biệt, `bal` về 0 đúng tại cuối mỗi khối.
- `s[start + 1:i]`: phần trong khối (bỏ `1` đầu và `0` cuối) — vẫn là chuỗi đặc biệt (hoặc rỗng), nên đệ quy hợp lệ.
- Chuỗi rỗng: vòng `for` không chạy, trả về `''` — trường hợp cơ sở tự nhiên.
- `blocks.sort(reverse=True)`: sắp giảm dần theo thứ tự từ điển.

## 10. Vì sao thuật toán đúng

**(a) Bất biến cấu trúc.** Gọi cây của `s` là `T(s)` (mục 4.3). Mọi bước đi chỉ hoán đổi hai dãy con anh em liên tiếp ở một nút → cây sau bước đi **đồng cấu** (cùng hình dạng nếu bỏ qua thứ tự con) với cây trước. Lý do: một chuỗi con đặc biệt `X = s[i:j]` có `bal` ≥ 0 bên trong và về 0 ở cuối, nên tầng (độ sâu) tại `i` bằng tầng tại `j`, và không có vị trí nào trong `X` xuống thấp hơn tầng đó → `X` là dãy các cây con liên tiếp của cùng một nút.

**(b) Mọi sắp xếp lại con đều đạt được.** Đổi chỗ hai khối anh em kề nhau là một bước đi hợp lệ (cả hai đều đặc biệt). Mọi hoán vị là tích các phép đổi chỗ kề nhau → sắp được con của mỗi nút theo thứ tự bất kỳ, độc lập giữa các nút.

**(c) Thứ tự tối ưu.** Ở mỗi nút, sau khi các cây con đã được tối ưu (quy nạp theo chiều cao), ta cần ghép các chuỗi `B1, ..., Bm` sao cho lớn nhất. Vì không khối nào là tiền tố của khối khác, phép so sánh `X + Y` vs `Y + X` được quyết định ở vị trí khác nhau đầu tiên của `X` và `Y` → sắp giảm dần là tối ưu (lập luận trao đổi: nếu tồn tại cặp kề `Bi < Bi+1`, đổi chỗ làm chuỗi lớn hơn).

**(d) Tối ưu cục bộ ⇒ toàn cục.** Phần trong của một khối nằm giữa `1` và `0` cố định; làm phần trong lớn hơn không bao giờ làm chuỗi lớn nhỏ đi; và các khối có độ dài cố định nên so sánh hai cách sắp không bị ảnh hưởng chéo.

## 11. Phân tích Time/Space

| Cách | Time | Space |
|---|---|---|
| 1. BFS vét cạn | mũ / giai thừa theo số khối, `O(n^4)` mỗi trạng thái | số trạng thái × `n` |
| 2. Đệ quy | `O(n^2 log n)` xấu nhất (chuỗi lồng sâu) | `O(n^2)` chuỗi con, đệ quy `O(n)` |
| 3. Stack | `O(n^2 log n)` | `O(n^2)` |
| 4. Cây + chuẩn hoá | `O(n^2 log n)` | `O(n^2)` |

Với `n ≤ 50` tất cả cách 2–4 chạy dưới 1 ms.

In [18]:
import random, time

def random_special(n_pairs, rnd=random):
    """Sinh ngẫu nhiên chuỗi đặc biệt có n_pairs cặp (không đều nhưng đủ dùng)."""
    res, opened, closed = [], 0, 0
    while closed < n_pairs:
        if opened < n_pairs and (opened == closed or rnd.random() < 0.5):
            res.append('1'); opened += 1
        else:
            res.append('0'); closed += 1
    return ''.join(res)

random.seed(761)
big = [random_special(25) for _ in range(2000)]
for name, f in [("rec", solve_rec), ("stack", solve_stack), ("tree", solve_tree)]:
    t0 = time.perf_counter()
    out = [f(x) for x in big]
    print(f"{name:6} {1000 * (time.perf_counter() - t0):7.1f} ms cho 2000 chuỗi dài 50")

rec       43.3 ms cho 2000 chuỗi dài 50
stack     13.3 ms cho 2000 chuỗi dài 50


tree      36.0 ms cho 2000 chuỗi dài 50


## 12. Bộ test

In [19]:
methods = {
    "rec": solve_rec,
    "stack": solve_stack,
    "tree": solve_tree,
    "SolutionRec": lambda s: SolutionRec().makeLargestSpecial(s),
    "SolutionStack": lambda s: SolutionStack().makeLargestSpecial(s),
    "SolutionTree": lambda s: SolutionTree().makeLargestSpecial(s),
    "Solution": lambda s: Solution().makeLargestSpecial(s),
}

# 12.1 Ví dụ đề (+ brute)
for s, exp in EXAMPLES:
    assert brute(s) == exp
    for name, f in methods.items():
        assert f(s) == exp, (name, s)

# 12.2 Edge cases
edge = [
    ("1010", "1010"),
    ("1100", "1100"),
    ("101100", "110010"),
    ("10101010", "10101010"),
    ("11110000", "11110000"),
    ("10110010", "11001010"),
]
for s, exp in edge:
    assert is_special(s)
    for name, f in methods.items():
        assert f(s) == exp, (name, s, f(s))
long_s = "10" * 25
assert Solution().makeLargestSpecial(long_s) == long_s
deep = "1" * 25 + "0" * 25
assert Solution().makeLargestSpecial(deep) == deep
print("Ví dụ + edge cases: OK")

Ví dụ + edge cases: OK


In [20]:
# 12.3 Đối chiếu brute force trên MỌI chuỗi đặc biệt độ dài <= 12
from itertools import product

def all_special(n_pairs):
    return [''.join(p) for p in product('10', repeat=2 * n_pairs) if is_special(''.join(p))]

cnt = 0
for k in range(1, 7):
    for s in all_special(k):
        b = brute(s)
        for name, f in methods.items():
            assert f(s) == b, (name, s, f(s), b)
        cnt += 1
print("Đã kiểm tra toàn bộ", cnt, "chuỗi đặc biệt độ dài <= 12")

# 12.4 Ngẫu nhiên (seed) dài hơn: kết quả là đặc biệt, cùng dạng chuẩn, và là điểm bất động
random.seed(2025)
for _ in range(500):
    s = random_special(random.randint(1, 25))
    r = Solution().makeLargestSpecial(s)
    assert is_special(r) and sorted(r) == sorted(s)
    assert r >= s and Solution().makeLargestSpecial(r) == r
    assert len({f(s) for f in methods.values()}) == 1
print("Ngẫu nhiên: OK")

Đã kiểm tra toàn bộ 196 chuỗi đặc biệt độ dài <= 12


Ngẫu nhiên: OK


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Sắp khối nhưng **quên đệ quy** vào bên trong

**Code sai:**

In [21]:
def wrong_no_recursion(s):
    blocks = split_primitives(s)
    blocks.sort(reverse=True)            # SAI: chỉ sắp tầng ngoài cùng
    return ''.join(blocks)

s = "11011000"
print("sai:", wrong_no_recursion(s), " đúng:", solve_rec(s))

sai: 11011000  đúng: 11100100


**Vì sao sai:** `"11011000"` chỉ có **một** khối ở tầng ngoài; mọi cải thiện nằm bên trong. Phải tối ưu từng tầng.

**Code đúng:** `'1' + solve(inner) + '0'` cho mỗi khối (Cách 2).

### Lỗi 2 — Đệ quy trên cả khối (không bỏ `1` đầu và `0` cuối) → đệ quy vô hạn

**Code sai:**

In [22]:
import sys

def wrong_infinite(t):
    blocks, bal, start = [], 0, 0
    for i, c in enumerate(t):
        bal += 1 if c == '1' else -1
        if bal == 0:
            blocks.append(wrong_infinite(t[start:i + 1]))   # SAI: gọi lại trên chính khối
            start = i + 1
    blocks.sort(reverse=True)
    return ''.join(blocks)

old = sys.getrecursionlimit()
sys.setrecursionlimit(200)
try:
    wrong_infinite("10")
except RecursionError:
    print("RecursionError: chuỗi '10' gọi lại chính nó mãi")
finally:
    sys.setrecursionlimit(old)

RecursionError: chuỗi '10' gọi lại chính nó mãi


**Vì sao sai:** khối duy nhất của `"10"` là `"10"` → gọi lại với input y hệt → không bao giờ dừng.

**Code đúng:** đệ quy trên `t[start + 1:i]`, rồi bọc lại `'1' + ... + '0'`.

### Lỗi 3 — Tham lam đổi chỗ "một lần" hoặc sắp **từng ký tự**

**Code sai:**

In [23]:
def wrong_sort_chars(s):
    return ''.join(sorted(s, reverse=True))      # SAI: "1111...0000" có thể không đạt được

def wrong_one_swap(s):
    best = s
    for v in moves(s):                           # SAI: chỉ thử một bước
        best = max(best, v)
    return best

s = "1010110100"
print("sắp ký tự:", wrong_sort_chars(s), "| một bước:", wrong_one_swap(s), "| đúng:", solve_rec(s))

sắp ký tự: 1111100000 | một bước: 1101001010 | đúng: 1101001010


**Vì sao sai:** sắp ký tự bỏ qua ràng buộc cấu trúc (độ sâu cây không đổi được — `"1010"` không bao giờ thành `"1100"`). Một bước đi thì chưa đủ, thường cần nhiều bước ở nhiều tầng.

**Code đúng:** đệ quy + sắp khối.

### Lỗi 4 — Sắp tăng dần (quên `reverse=True`)

In [24]:
def wrong_ascending(t):
    blocks, bal, start = [], 0, 0
    for i, c in enumerate(t):
        bal += 1 if c == '1' else -1
        if bal == 0:
            blocks.append('1' + wrong_ascending(t[start + 1:i]) + '0')
            start = i + 1
    blocks.sort()                                # SAI: ra chuỗi NHỎ nhất
    return ''.join(blocks)

print("sai:", wrong_ascending("11011000"), " đúng:", solve_rec("11011000"))

sai: 11011000  đúng: 11100100


**Vì sao sai:** sắp tăng cho chuỗi **nhỏ nhất** (thú vị là cũng là một bài toán hợp lệ!). **Code đúng:** `sort(reverse=True)`.

## 14. Mẹo & nhận dạng pattern

- **"Số 1 bằng số 0, mọi tiền tố 1 ≥ 0"** → chuỗi ngoặc đúng / đường Dyck → nghĩ đến **cây** và **stack**.
- **Cắt khi `bal == 0`** → tách khối nguyên thuỷ (giống *1021. Remove Outermost Parentheses*, *856. Score of Parentheses*).
- **Được hoán đổi khối kề nhau tuỳ ý** → được **hoán vị tuỳ ý** → chỉ còn bài **sắp xếp**.
- **Ghép chuỗi để lớn nhất**: nếu không chuỗi nào là tiền tố của chuỗi khác, sắp giảm dần là đủ; tổng quát phải dùng khoá so sánh `a + b > b + a` (*179. Largest Number*).
- **Dạng chuẩn của cây** (AHU) → so sánh cây đồng cấu.

## 15. Biến thể

1. **Chuỗi nhỏ nhất** → sắp tăng dần.
2. **Kiểm tra hai chuỗi đặc biệt có biến đổi được thành nhau không** → so sánh dạng chuẩn.
3. **Đếm số chuỗi khác nhau đạt được** → tích, tại mỗi nút, số hoán vị phân biệt của các con: `m! / (c1! c2! ...)` (chia cho giai thừa số con giống nhau).
4. **Bảng chữ ngoặc nhiều loại** (`()`, `[]`) → vẫn cây, nhưng nhãn nút có loại ngoặc.
5. **Ghép số lớn nhất** (*179*) → khoá so sánh `a + b`.

In [25]:
from math import factorial
from collections import Counter

def count_reachable(s):
    """Biến thể 3: số chuỗi khác nhau đạt được."""
    def go(t):
        blocks, ways, bal, start = [], 1, 0, 0
        for i, c in enumerate(t):
            bal += 1 if c == '1' else -1
            if bal == 0:
                w, canon_inner = go(t[start + 1:i])
                ways *= w
                blocks.append('1' + canon_inner + '0')
                start = i + 1
        perm = factorial(len(blocks))
        for v in Counter(blocks).values():
            perm //= factorial(v)
        blocks.sort(reverse=True)
        return ways * perm, ''.join(blocks)
    return go(s)[0]

for s in ["11011000", "1010110100", "101100110100"]:
    a, b = count_reachable(s), len(bfs_demo(s, moves))
    print(s, a, b)
    assert a == b

11011000 2 2
1010110100 3 3
101100110100 6 6


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS vét cạn | Duyệt mọi chuỗi đạt được | Mũ | Mũ | Kiểm thử chuỗi ngắn |
| 2. Đệ quy | Tách khối, đệ quy phần trong, sắp giảm | `O(n^2 log n)` | `O(n^2)` | Nộp bài — ngắn nhất |
| 3. Stack | Mở tầng khi `1`, đóng & sắp khi `0` | `O(n^2 log n)` | `O(n^2)` | Tránh đệ quy |
| 4. Cây + dạng chuẩn | Dựng cây rồi chuẩn hoá AHU | `O(n^2 log n)` | `O(n^2)` | Khi cần so sánh cấu trúc, đếm |

## 17. Bài tập

**Bài 1 (đoán output).** `makeLargestSpecial("1010110100")` = ?

**Bài 2 (tính tay).** Phân rã `"110100110010"` thành các khối nguyên thuỷ ở tầng ngoài và tầng trong; viết cây.

**Bài 3 (điền TODO).** Hoàn thiện `make_smallest_special(s)` trả về chuỗi đặc biệt **nhỏ nhất** đạt được.

**Bài 4 (tự viết lại).** Viết lại Cách 3 (stack) mà không dùng `sorted` có sẵn — tự cài sắp xếp chèn (insertion sort) giảm dần.

**Bài 5 (mini challenge).** Viết `can_transform(a, b)` trả `True` nếu chuỗi đặc biệt `a` biến đổi được thành `b`; đối chiếu với BFS.

In [26]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def make_smallest_special(s):
    # TODO: như Cách 2 nhưng sắp tăng dần
    pass

# Bài 4 — TODO
def solve_stack_insertion(s):
    # TODO: thay inner.sort(reverse=True) bằng insertion sort tự viết
    pass

# Bài 5 — TODO
def can_transform(a, b):
    # TODO: so sánh dạng chuẩn
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: tầng ngoài: `10`, `10`, `110100`.
- Mức 2: bên trong `110100` là `1 + "1010" + 0`, `1010` đã tối ưu → `110100`.
- Mức 3: sắp `[110100, 10, 10]` → `"1101001010"`.

**Bài 2.**
- Mức 1: tính `bal` từng vị trí: 1,2,1,2,1,**0**,1,2,1,**0**,1,**0** → về 0 tại chỉ số 5, 9, 11.
- Mức 2: tầng ngoài có 3 khối: `110100`, `1100`, `10`.
- Mức 3: `110100` = `1 + [10, 10] + 0`; `1100` = `1 + [10] + 0`; `10` = `1 + [] + 0`. Cây: gốc có 3 con; con thứ nhất có 2 lá, con thứ hai có 1 lá, con thứ ba là lá.

**Bài 3.**
- Mức 1: giữ nguyên khung đệ quy.
- Mức 2: đổi `reverse=True` thành mặc định.
- Mức 3: kiểm tra bằng `min` trong BFS.

**Bài 4.**
- Mức 1: duyệt từng phần tử, chèn vào vị trí đúng.
- Mức 2: so sánh `x > arr[j]` để giảm dần.
- Mức 3: so sánh kết quả với `solve_stack`.

**Bài 5.**
- Mức 1: hai chuỗi biến đổi được thành nhau ⇔ cùng cây (bỏ qua thứ tự con).
- Mức 2: dạng chuẩn = `solve(a)`.
- Mức 3: `len(a) == len(b) and solve(a) == solve(b)`.

## 19. Đáp án tham khảo

In [27]:
# Bài 1
assert Solution().makeLargestSpecial("1010110100") == "1101001010" == brute("1010110100")

# Bài 2
assert split_primitives("110100110010") == ["110100", "1100", "10"]
assert split_primitives("1010") == ["10", "10"]

# Bài 3
def make_smallest_special_ans(s):
    blocks, bal, start = [], 0, 0
    for i, c in enumerate(s):
        bal += 1 if c == '1' else -1
        if bal == 0:
            blocks.append('1' + make_smallest_special_ans(s[start + 1:i]) + '0')
            start = i + 1
    blocks.sort()
    return ''.join(blocks)

def brute_min(s):
    return min(bfs_demo(s, moves))

for k in range(1, 6):
    for s in all_special(k):
        assert make_smallest_special_ans(s) == brute_min(s), s

# Bài 4
def solve_stack_insertion_ans(s):
    def ins_sort_desc(arr):
        for i in range(1, len(arr)):
            x, j = arr[i], i - 1
            while j >= 0 and x > arr[j]:
                arr[j + 1] = arr[j]
                j -= 1
            arr[j + 1] = x
        return arr
    stack = [[]]
    for c in s:
        if c == '1':
            stack.append([])
        else:
            inner = ins_sort_desc(stack.pop())
            stack[-1].append('1' + ''.join(inner) + '0')
    return ''.join(ins_sort_desc(stack[0]))

random.seed(8)
for _ in range(300):
    s = random_special(random.randint(1, 20))
    assert solve_stack_insertion_ans(s) == solve_stack(s)

# Bài 5
def can_transform_ans(a, b):
    return len(a) == len(b) and solve_rec(a) == solve_rec(b)

for k in range(1, 5):
    ss = all_special(k)
    for a in ss:
        reach = bfs_demo(a, moves)
        for b in ss:
            assert can_transform_ans(a, b) == (b in reach), (a, b)
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template:**

```python
def solve(s):
    blocks, bal, start = [], 0, 0
    for i, c in enumerate(s):
        bal += 1 if c == '1' else -1
        if bal == 0:                                   # hết một khối 1 A 0
            blocks.append('1' + solve(s[start+1:i]) + '0')
            start = i + 1
    return ''.join(sorted(blocks, reverse=True))       # khối lớn trước
```

**Checklist trước khi submit:**

- [ ] Cắt khối khi `bal == 0`.
- [ ] Đệ quy trên **phần trong** `s[start+1:i]`, không phải cả khối.
- [ ] Bọc lại `'1' + ... + '0'`.
- [ ] Sắp **giảm dần** (`reverse=True`).
- [ ] Chuỗi rỗng trả `''` (trường hợp cơ sở).
- [ ] Đã thử `"10"`, `"1010"`, chuỗi lồng sâu `"1"*25 + "0"*25`.
- [ ] Tên method đúng: `makeLargestSpecial(self, s)`.